In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, hamming_loss
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline

random_seed = 133

In [6]:
df = pd.read_csv('../../annotation/email_understanding/annotation_output/full/df4model.csv')

df = df.dropna()

In [7]:
sum(df['label_Sender Expectation'].isna())

0

In [8]:
columns = list(df.columns)

label_columns = [x for x in columns if 'label' in x]

labels = df[label_columns].values

In [9]:
label_columns

['label_Act:::Thank you/Welcome',
 'label_Act:::Commit/Agree',
 'label_Act:::Request',
 'label_Act:::Deliver/Informative',
 'label_Act:::Amend',
 'label_Act:::Remind',
 'label_Act:::Refuse',
 'label_Act:::Introduction',
 'label_Act:::Propose',
 'label_spam',
 'label_Sender Expectation']

In [10]:
labels

array([[0., 0., 1., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 1.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 1.]])

In [11]:
text_samples = df['cleaned_text'].to_numpy()

In [12]:
X_train, X_test, y_train, y_test = train_test_split(text_samples, labels, test_size=0.2, random_state=random_seed)

In [13]:
logistic_model = LogisticRegression()
multi_label_model = MultiOutputClassifier(logistic_model)


In [14]:
pipeline = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 2))),
    ('classifier', multi_label_model)
])

# Define the parameter grid to search over
param_grid = {
    'classifier__estimator__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__estimator__max_iter': [100, 200, 300]
}

In [15]:
# Define a custom scorer for Hamming loss
hamming_loss_scorer = make_scorer(hamming_loss, greater_is_better=False)

# Instantiate the GridSearchCV object with Hamming loss as the scoring metric
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring=hamming_loss_scorer, verbose = 2)

# Perform grid search on the data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=100; total time=   2.9s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=100; total time=   2.1s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=100; total time=   2.0s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=100; total time=   2.4s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=100; total time=   1.3s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=200; total time=   2.2s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=200; total time=   1.9s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=200; total time=   3.1s
[CV] END classifier__estimator__C=0.001, classifier__estimator__max_iter=200; total time=   2.2s
[CV] END classifier__estimator__C=0.001, classifier__estimator__ma

[CV] END classifier__estimator__C=100, classifier__estimator__max_iter=300; total time=   4.9s
[CV] END classifier__estimator__C=100, classifier__estimator__max_iter=300; total time=   6.0s
[CV] END classifier__estimator__C=100, classifier__estimator__max_iter=300; total time=   5.3s
[CV] END classifier__estimator__C=100, classifier__estimator__max_iter=300; total time=   2.5s


/opt/anaconda/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
18 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
18 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda/lib/python3.10/site-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/opt/anaconda/lib/python3.10/site-packages/sklearn/multioutput.py", line 450, in fit
    super().fit(X, Y, sample_weight, **fit_params)
  File "/opt/an

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vectorizer',
                                        CountVectorizer(ngram_range=(1, 2))),
                                       ('classifier',
                                        MultiOutputClassifier(estimator=LogisticRegression()))]),
             param_grid={'classifier__estimator__C': [0.001, 0.01, 0.1, 1, 10,
                                                      100],
                         'classifier__estimator__max_iter': [100, 200, 300]},
             scoring=make_scorer(hamming_loss, greater_is_better=False),
             verbose=2)

In [18]:
y_pred = grid_search.best_estimator_.predict(X_test)

In [21]:
grid_search.best_params_

{'classifier__estimator__C': 0.001, 'classifier__estimator__max_iter': 100}

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {}

for class_index in range(len(label_columns)):
    metrics[label_columns[class_index]] = {} 
    accuracy = accuracy_score(y_test[:, class_index], y_pred[:, class_index])
    precision = precision_score(y_test[:, class_index], y_pred[:, class_index], average='micro')  # For multi-label classification
    recall = recall_score(y_test[:, class_index], y_pred[:, class_index], average='micro')  # For multi-label classification
    f1 = f1_score(y_test[:, class_index], y_pred[:, class_index], average='micro')  # For multi-label classification
    
    metrics[label_columns[class_index]]['accuracy'] = accuracy
    metrics[label_columns[class_index]]['precision'] = precision
    metrics[label_columns[class_index]]['recall'] = recall
    metrics[label_columns[class_index]]['f1'] = f1

{'label_Act:::Thank you/Welcome': {'accuracy': 0.9862068965517241, 'precision': 0.9862068965517241, 'recall': 0.9862068965517241, 'f1': 0.9862068965517241}, 'label_Act:::Commit/Agree': {'accuracy': 0.9241379310344827, 'precision': 0.9241379310344827, 'recall': 0.9241379310344827, 'f1': 0.9241379310344827}, 'label_Act:::Request': {'accuracy': 0.6344827586206897, 'precision': 0.6344827586206897, 'recall': 0.6344827586206897, 'f1': 0.6344827586206897}, 'label_Act:::Deliver/Informative': {'accuracy': 0.7793103448275862, 'precision': 0.7793103448275862, 'recall': 0.7793103448275862, 'f1': 0.7793103448275862}, 'label_Act:::Amend': {'accuracy': 0.9862068965517241, 'precision': 0.9862068965517241, 'recall': 0.9862068965517241, 'f1': 0.9862068965517241}, 'label_Act:::Remind': {'accuracy': 0.993103448275862, 'precision': 0.993103448275862, 'recall': 0.993103448275862, 'f1': 0.993103448275862}, 'label_Act:::Refuse': {'accuracy': 0.993103448275862, 'precision': 0.993103448275862, 'recall': 0.99310

In [28]:
hamming = hamming_loss(y_test, y_pred)

hamming

0.09968652037617555